# LangGraph Module · Day 04 — Tool Calling & Tool Nodes

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Implement **`@tool`** decorator functions (web search, calculator, file reader)
- [ ] Create a **`ToolNode`** and bind tools to a **ReAct agent**
- [ ] Test tool execution **+ error handling** flows

> Builds on Day 03 (state & memory) and today's Python topic (`@tool` internals + `@retry`).

## 0. Setup — load the API key

Runs against **Google Gemini** (needs `GOOGLE_API_KEY` in a `.env` at the project root). Tool *definitions* and `ToolNode` work without a key; the model calls that pick tools need one, so those cells are guarded by `HAS_KEY`.

Install (if needed): `pip install langgraph langchain-google-genai python-dotenv`

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
HAS_KEY = bool(os.getenv("GOOGLE_API_KEY"))
print("langgraph tool-calling notebook")
print("GOOGLE_API_KEY:", "loaded ✅" if HAS_KEY else "MISSING ❌ (model cells will be skipped)")

langgraph tool-calling notebook
GOOGLE_API_KEY: loaded ✅


## 1. Why tools — the LLM can't actually *do* things

An LLM predicts text. It can't reliably multiply large numbers, fetch a live web page, or read a file off disk. **Tool calling** fixes this: you give the model a set of functions, it decides *which* to call and *with what arguments*, your code runs the real function, and the result goes back to the model.

The loop (this is **ReAct** — Reason + Act):

```
user question -> [model] -> "call calculator(a=1234, b=5678)"
             -> [your code runs it] -> 7024652
             -> [model] -> "The answer is 7024652."
```

## 2. Define tools with `@tool`

The real `@tool` from `langchain_core.tools` does exactly what yesterday's `MiniTool` did: reads the **name**, **docstring** (→ description the LLM sees), and **type hints** (→ input schema). Write the docstring for the *model*, not just humans.

In [2]:
from langchain_core.tools import tool

@tool
def calculator(a: float, b: float) -> float:
    """Multiply two numbers a and b, and return the product."""
    return a * b

@tool
def web_search(query: str) -> str:
    """Search the web for a query and return a short text snippet."""
    # Mocked so the notebook is deterministic and offline
    fake = {
        "capital of france": "Paris is the capital of France.",
        "langgraph": "LangGraph is a library for building stateful, multi-actor LLM apps.",
    }
    return fake.get(query.lower().strip(), f"No results found for {query!r}.")

@tool
def file_reader(path: str) -> str:
    """Read a text file at the given path and return its contents."""
    # Mocked file system so the demo needs no real files
    files = {"notes.txt": "Buy milk. Ship the invoice agent. Call the bank."}
    if path not in files:
        raise FileNotFoundError(f"no such file: {path}")
    return files[path]

TOOLS = [calculator, web_search, file_reader]
for t in TOOLS:
    print(f"{t.name:12} args={list(t.args)}  desc={t.description!r}")

calculator   args=['a', 'b']  desc='Multiply two numbers a and b, and return the product.'
web_search   args=['query']  desc='Search the web for a query and return a short text snippet.'
file_reader  args=['path']  desc='Read a text file at the given path and return its contents.'


☝️ Each tool is now a self-describing object: `name`, `args` (from the hints), and `description` (from the docstring). Invoke one directly to confirm it's just a function underneath:

In [3]:
print("calculator.invoke ->", calculator.invoke({"a": 6, "b": 7}))
print("web_search.invoke ->", web_search.invoke({"query": "capital of france"}))

calculator.invoke -> 42.0
web_search.invoke -> Paris is the capital of France.


## 3. `bind_tools` — let the model *choose* a tool

`model.bind_tools(TOOLS)` hands the tool schemas to the model. The model no longer just returns text — it can return an `AIMessage` whose **`.tool_calls`** lists the tools it wants to run and the arguments it picked. It does **not** run them; that's your job (section 5).

In [4]:
if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI

    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    model_with_tools = model.bind_tools(TOOLS)

    msg = model_with_tools.invoke("What is 231 times 47?")
    print("content     :", repr(msg.content))        # often empty when a tool is called
    print("tool_calls  :", msg.tool_calls)           # <- the model's decision
else:
    print("skipped (no API key) — model would return an AIMessage with .tool_calls")

content     : ''
tool_calls  : [{'name': 'calculator', 'args': {'a': 231, 'b': 47}, 'id': '843a0307-a010-4c24-a675-f950e9053cea', 'type': 'tool_call'}]


☝️ The model read the tool descriptions, decided `calculator` fits, and filled in `a` and `b` from the question. `content` is usually empty on a tool-calling turn — the model is "acting", not "answering" yet.

## 4. `ToolNode` — a prebuilt node that runs the tool calls

You *could* loop over `.tool_calls` yourself. LangGraph ships **`ToolNode`**: give it your tools, and it reads the last `AIMessage`'s `tool_calls`, executes each tool, and returns the results as **`ToolMessage`** objects appended to `messages`.

One thing to know for LangGraph 1.x: `ToolNode` is a **graph node**, not a standalone function — it expects a graph runtime, so you can't call `ToolNode(...).invoke(...)` bare. Wrap it in a tiny one-node graph. No API key needed — we feed it a hand-crafted `AIMessage`.

In [5]:
from typing import Annotated, TypedDict
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage

class ToolState(TypedDict):
    messages: Annotated[list, add_messages]

def tools_only_graph(tool_node: ToolNode):
    """Wrap a ToolNode in START -> tools -> END so we can run it offline."""
    b = StateGraph(ToolState)
    b.add_node("tools", tool_node)
    b.add_edge(START, "tools")
    b.add_edge("tools", END)
    return b.compile()

demo = tools_only_graph(ToolNode(TOOLS))

# Simulate what the model produced in section 3 (so this runs offline)
fake_ai = AIMessage(
    content="",
    tool_calls=[{"name": "calculator", "args": {"a": 231, "b": 47}, "id": "call_1"}],
)

result = demo.invoke({"messages": [fake_ai]})
tmsg = result["messages"][-1]
print("type   :", type(tmsg).__name__)     # ToolMessage
print("name   :", tmsg.name)               # calculator
print("content:", tmsg.content)            # '10857.0'  <- the real product

type   : ToolMessage
name   : calculator
content: 10857.0


## 5. Build the ReAct agent — the shortcut: `create_react_agent`

`create_react_agent(model, tools)` wires the whole loop for you: model node ⇄ ToolNode, with a conditional edge that keeps looping while the model asks for tools and stops when it produces a final answer. This is the fastest way to a working tool-calling agent.

In [ ]:
if HAS_KEY:
    #from langgraph.prebuilt import create_react_agent
    from langchain.agents import create_agent

    #agent = create_react_agent(model, TOOLS)
    agent = create_agent(model, TOOLS)

    out = agent.invoke({"messages": [("user", "What is 231 * 47?")]})
    for m in out["messages"]:
        role = type(m).__name__.replace("Message", "")
        extra = f" tool_calls={m.tool_calls}" if getattr(m, "tool_calls", None) else ""
        print(f"[{role:9}] {m.content!r}{extra}")
else:
    print("skipped (no API key)")

[Human    ] 'What is 231 * 47?'
[AI       ] '' tool_calls=[{'name': 'calculator', 'args': {'b': 47, 'a': 231}, 'id': 'c87bac88-77eb-49f5-ae32-98e07d1f16ba', 'type': 'tool_call'}]
[Tool     ] '10857.0'
[AI       ] '231 * 47 = 10857'


☝️ Read the trace top to bottom: **Human** asks → **AI** emits a `calculator` tool_call → **Tool** returns `10857` → **AI** writes the final sentence. That back-and-forth *is* ReAct, and the conditional edge is what let it loop then stop.

## 6. Build it by hand — see what `create_react_agent` hides

The shortcut is great, but you should know the wiring. The graph is two nodes and one conditional edge:

- **`agent`** node — calls `model_with_tools`, returns its `AIMessage`.
- **`tools`** node — the `ToolNode` from section 4.
- **conditional edge** — after `agent`, use LangGraph's prebuilt **`tools_condition`**: if the last message has `tool_calls` → go to `tools`; else → `END`.
- **edge** `tools -> agent` — feed tool results back so the model can continue.

This is the exact skeleton `create_react_agent` builds.

In [8]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

if HAS_KEY:
    def agent_node(state: AgentState) -> dict:
        # call the model; it may ask for a tool or give a final answer
        return {"messages": [model_with_tools.invoke(state["messages"])]}

    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", tools_condition)   # agent -> tools OR END
    builder.add_edge("tools", "agent")                        # loop tool result back
    react = builder.compile()

    print(react.get_graph().draw_ascii())
else:
    print("skipped (no API key)")

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [ ]:
if HAS_KEY:
    out = react.invoke({"messages": [("user", "Read notes.txt and tell me the first task.")]})
    for m in out["messages"]:
        role = type(m).__name__.replace("Message", "")
        extra = f" tool_calls={m.tool_calls}" if getattr(m, "tool_calls", None) else ""
        print(f"[{role:9}] {m.content!r}")
else:
    print("skipped (no API key)")

[Human    ] 'Read notes.txt and tell me the first task.'
[AI       ] ''
[Tool     ] 'Buy milk. Ship the invoice agent. Call the bank.'
[AI       ] [{'type': 'text', 'text': 'The first task is: Buy milk.', 'extras': {'signature': 'CqACAQw51sf6+nSck5lvdStCQe74shhvEgrc+aPRhB5oefQ+CyJLjbyZD0pVG3YL20YN9YtEdJe3G9byQ/OgmlCdIipUISj2URvkms6eEyVtq3Y28eAB3+Hk0BMA3TkSrwCPCWcpTf954E9iUwD0lquNzCTQNd4xAlqIH2/J6ZVs1/aCiJNjNG9U4ihV29dPBdau1speHBjruN/GCjeWM5EHMSdQo3B8tbiJDAdOdnlcOj8cU7YcdXBbjLKUHVFwehr/4Dl2Yuil+31udSqnWfkBh2q6psY0Wie792M45ncZqzKbYmWIcr5nFJyWjtNtHQ8Fv91x65yhD3Ovil7CB+gwJhoBXUqXc2JLGkD93Bu8R1vnX74SBc7jOxjEe8ulUEuJ'}}]


☝️ Same behaviour as `create_react_agent`, but now you can see the seam: `tools_condition` is the router, and `tools -> agent` is the loop. Swap in your own router here when you need custom control (e.g. a human-approval gate — that's Day 05).

## 7. Error handling — tools fail, and the agent should survive

`file_reader` raises `FileNotFoundError` on a bad path. Pass **`handle_tool_errors=True`** to `ToolNode` and it **catches** the exception, returning it to the model as a `ToolMessage` with `status="error"` — so the model can react (retry, apologise, try another path) instead of crashing the graph.

> Without that flag, a raw tool exception (other than a bad-arguments error) propagates and stops the run. Turn it on for agent robustness.

In [14]:
from langchain_core.messages import AIMessage

# Force a failing tool call: a path that doesn't exist
bad_ai = AIMessage(
    content="",
    tool_calls=[{"name": "file_reader", "args": {"path": "missing.txt"}, "id": "call_err"}],
)

# handle_tool_errors=True -> the error is captured and returned; graph keeps running
safe = tools_only_graph(ToolNode(TOOLS, handle_tool_errors=True))
res = safe.invoke({"messages": [bad_ai]})
err_msg = res["messages"][-1]
print("status :", err_msg.status)          # 'error'
print("content:", err_msg.content)         # the exception text, handed back to the model

status : error
content: Error: FileNotFoundError('no such file: missing.txt')
 Please fix your mistakes.


Combine this with **yesterday's `@retry`** for network-flaky tools: decorate the tool body so transient failures retry *before* they ever become a `ToolMessage` error. Belt (retry) and braces (ToolNode capture).

In [15]:
import functools, time

def retry(times=3, delay=0.02):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*a, **k):
            last = None
            for i in range(times):
                try:
                    return func(*a, **k)
                except Exception as e:
                    last, _ = e, time.sleep(delay)
            raise last
        return wrapper
    return deco

_hits = {"n": 0}

@tool
def flaky_search(query: str) -> str:
    """Search a flaky service that fails intermittently."""
    return _inner(query)

@retry(times=4)
def _inner(query: str) -> str:
    _hits["n"] += 1
    if _hits["n"] < 3:
        raise ConnectionError("transient 503")
    return f"result for {query!r} after {_hits['n']} tries"

print(flaky_search.invoke({"query": "langgraph"}))   # retries under the hood, then succeeds

result for 'langgraph' after 3 tries


☝️ The `@retry` sits *inside* the tool, so ToolNode never even sees the transient failures — it only sees the eventual success (or a real, permanent error worth surfacing to the model).

## 8. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **`@tool`** | wraps a function into a self-describing tool (name + docstring + hints) | lets the LLM know what it can call and how |
| **`bind_tools`** | attaches tool schemas to a model | model returns `.tool_calls` instead of guessing |
| **`.tool_calls`** | list of `{name, args, id}` on the `AIMessage` | the model's *decision* — not yet executed |
| **`ToolNode`** | prebuilt node that runs the tool calls | turns `tool_calls` into `ToolMessage` results |
| **`tools_condition`** | prebuilt router: tool_calls → `tools`, else → `END` | the branch that makes the ReAct loop loop/stop |
| **`create_react_agent`** | one call that wires model ⇄ ToolNode | fastest path to a working tool-calling agent |
| **error handling** | ToolNode captures tool exceptions as error `ToolMessage`s | agent survives failures; pair with `@retry` inside the tool |

### Done today
- [x] `@tool` functions: calculator, web_search, file_reader
- [x] `ToolNode` + bound tools + ReAct agent (shortcut **and** hand-wired)
- [x] Tested tool execution **and** error-handling flows